In [1]:
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트 (Windows: 맑은 고딕, Mac: AppleGothic)
plt.rcParams["font.family"] = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "Noto Sans CJK KR")
plt.rcParams["axes.unicode_minus"] = False

#DATA_PATH = "listings.csv"
#OUT_PATH = "listings_clean.csv"

ROOM_TYPES = ["Entire home/apt", "Private room"]   # 호텔룸·다인실 제외
MIN_REVIEWS = 10                                    # 평점 신뢰도
MIN_AVAILABILITY = 1                                # 예약 가능일 1일 이상 (팀 합의 사항)
MAX_QUOTE_NIGHTS = 6                               # 가격 견적 6박 이하 (장기 할인 가격 제외) — 팀 합의 필요
TRIM_Q = 0.01                                       # 가격 상·하위 1% 제외


# 이름에 운영 중단이 명시된 숙소 (직접 확인, 6개)
INACTIVE_IDS = [13254774, 16564598, 19557335, 20552841, 51397574, 635527106503833045]
CENTER = (51.5074, -0.1278)                         # 도심 기준점: Charing Cross 부근

### 판단: 가격 상·하위 1% 제외 (£40 미만, £958 초과, 511개)

- 극단적인 가격은 적정 가격 모델을 왜곡하므로 제외한다. 기준값이 바뀌지 않도록 다른 필터를 모두 적용한 마지막에 자른다.
- 상위 1%: 2인용 원룸 £10,000처럼 예약을 막으려고 걸어둔 것으로 보이는 가격이 섞여 있다. 진짜 고급 대형 숙소도 일부 포함되지만(수용 인원 중앙값 6명), 수가 적어 영향은 작다고 판단한다.
- 하위 1%: 대부분 외곽 1인용 개인실로 실제로 싼 숙소다. 오류는 아니지만 비슷한 조건의 숙소도 싸므로 제외해도 결론 영향은 작다.


### 판단: 가격 조회 기간 6박 이하만 사용

- 숙소끼리 가격을 비교하려면 같은 기준의 가격이 필요하다.
- 그런데 가격은 숙소마다 다른 숙박 일수로 조회되었다 (대부분 최소 숙박일 기준).
- 조회 기간이 길면 장기 할인이 들어가 정상 가격 숙소가 "숨은 보석"으로 잘못 분류될 수 있다.
- 할인 적용 비율이 1~6박은 12.6~26.3%인데 7박부터 55% 이상으로 급증해, 6박 이하만 사용한다 (1,799개 제외).
- 96%가 최소 숙박일이 6박 이하이다.

## 1. 데이터 확인

In [2]:

df = pd.read_csv("listings.csv.gz", low_memory=False)
print(df.shape) 
print("수집일:", df["last_scraped"].min(), "~", df["last_scraped"].max())
print("자치구 수:", df["neighbourhood_cleansed"].nunique())

(91954, 90)
수집일: 2026-09-18 ~ 2026-09-23
자치구 수: 33


## 2. 가격 정리

**문제:** `price`는 `"$1,234.00"` 같은 문자열이고, 숙소마다 다른 날짜·숙박 일수로 조회한 견적의 1박 가격이다.

**하는 일**
- `$`와 쉼표를 지워 숫자로 변환 → `price_gbp`
- 견적 원문(`price_quote_raw`)에서 실제 통화 확인
- 체크아웃 − 체크인으로 가격 조회 기간 계산 → `quote_nights`

**결과**
- 통화는 전부 GBP → `$` 기호와 달리 **실제 단위는 파운드(£)**
- 가격 결측 **34.9%** → 아래 2-1에서 원인 확인
- 중앙값 £175, 평균 £248, 최댓값 £75,000 → 비싼 쪽으로 치우친 분포, 비정상 존재 가능성

In [3]:
# 변환 전 price 열 확인
print("자료형:", df["price"].dtype)
print("예시:", df["price"].dropna().head(5).tolist())


자료형: str
예시: ['$267.50', '$127.00', '$137.50', '$228.00', '$290.00']


In [4]:
df["price_gbp"] = df["price"].str.replace(r"[$,]", "", regex=True).astype(float)

currency = df["price_quote_raw"].dropna().str.extract(r'"currency": "(\w+)"')[0]
print("견적 통화:", currency.value_counts().to_dict())

df["quote_nights"] = (pd.to_datetime(df["price_quote_checkout_date"])
                      - pd.to_datetime(df["price_quote_checkin_date"])).dt.days
print("\n전체 가격 결측 비율:", round(df["price_gbp"].isna().mean(), 3))
print(df["price_gbp"].describe().round(1))

견적 통화: {'GBP': 60028}

전체 가격 결측 비율: 0.349
count    59905.0
mean       248.4
std        577.0
min         10.0
25%         99.7
50%        175.0
75%        279.8
max      75000.0
Name: price_gbp, dtype: float64


## 2-1 가격 결측은 무작위가 아니다

**확인한 것:** 가격이 없는 숙소가 어떤 숙소인지, 최근 1년 리뷰 유무로 나눠 가격 결측 비율을 비교했다.

**결과:**
- 리뷰 10개 이상 숙소의 26.7%가 가격이 없다.
- 최근 1년 리뷰가 없는 숙소는 70.5%, 있는 숙소는 14.3%가 가격이 없다.
- 최근 1년 리뷰가 없는 숙소는 마지막 리뷰가 중앙값 3.2년 전으로, 운영을 쉬고 있을 가능성이 높다.

**결론:** 가격 없는 숙소는 리뷰와 관계없이 **전부 제외**한다.
- 제외해도 현재 운영 중인 숙소 위주로 남으므로, "지금 받는 가격이 적정한가"라는 질문에 맞는 범위다.
- 가격은 분석 대상 자체라 다른 값으로 채우면 결과를 지어내는 셈이므로 채우지 않는다.

In [5]:
r10 = df[df["number_of_reviews"] >= MIN_REVIEWS]
print("리뷰 10개 이상 숙소의 가격 결측 비율:", round(r10["price_gbp"].isna().mean(), 3))
print(r10.groupby(r10["number_of_reviews_ltm"] == 0)["price_gbp"]
         .apply(lambda s: s.isna().mean()).round(3)
         .rename({False: "최근 1년 리뷰 있음", True: "최근 1년 리뷰 없음"}))

no_recent = r10[r10["number_of_reviews_ltm"] == 0]
years_since = (pd.to_datetime(no_recent["last_scraped"])
               - pd.to_datetime(no_recent["last_review"])).dt.days / 365
print("최근 1년 리뷰 없는 숙소:", len(no_recent), "개")
print("마지막 리뷰 이후 경과 연수 중앙값:", round(years_since.median(), 1))

리뷰 10개 이상 숙소의 가격 결측 비율: 0.267
number_of_reviews_ltm
최근 1년 리뷰 있음    0.143
최근 1년 리뷰 없음    0.705
Name: price_gbp, dtype: float64
최근 1년 리뷰 없는 숙소: 8525 개
마지막 리뷰 이후 경과 연수 중앙값: 3.2


## 2-2 조회 기간에 따라 가격과 할인이 달라진다 (집 전체 / 개인실)

**하는 일**
- 조회 기간별 1박 가격 중앙값을 비교한다.
- 견적 원문의 할인 금액(`discount_amount`)이 0보다 큰지로 할인 적용 비율을 구한다.

**결론**
- 두 유형 모두 **7박에서 할인 적용 비율이 55% 안팎으로 급증**한다. 주간 할인(7박 이상)이 시작되는 지점으로 보인다.
- 할인된 가격이 섞이면 정상 가격 숙소가 "숨은 보석"으로 잘못 분류될 수 있으므로, **조회 기간 6박 이하만 사용**한다.
- 개인실은 6박 이하에서도 조회 기간이 길수록 가격이 낮다(1박 £85 → 4~6박 £71).
- 이 구간은 할인 비율이 오히려 낮아(7.6~9.1%), 할인 때문이 아니다. 최소 숙박일이 긴 개인실이 원래 저렴한 방이기 때문으로 보인다.
- 가격 측정 문제가 아니라 숙소의 실제 차이이므로 빼지 않고, 조회 기간(`quote_nights`)을 모델 변수로 넘겨 반영한다.

In [6]:
# 조회 기간별 1박 가격 중앙값 (집 전체 / 개인실)
pr = df[df["room_type"].isin(ROOM_TYPES) & df["price_gbp"].notna()]
print(pr.groupby(["room_type", pd.cut(pr["quote_nights"], [0, 1, 3, 6, 30, 400])], observed=True)["price_gbp"]
        .agg(["median", "size"]).round(1))

                              median   size
room_type       quote_nights               
Entire home/apt (0, 1]         224.0  10745
                (1, 3]         230.5  19184
                (3, 6]         247.0   5337
                (6, 30]        206.4   4477
                (30, 400]      152.7    778
Private room    (0, 1]          85.0  10610
                (1, 3]          80.0   5980
                (3, 6]          70.5   1153
                (6, 30]         58.6   1215
                (30, 400]       51.2    127


In [7]:
# 할인 적용 비율 (집 전체 / 개인실)
def discount_amount(raw):
    if not isinstance(raw, str):
        return np.nan
    quote = json.loads(raw).get("quote") or {}
    d = quote.get("discount_amount")
    return float(d) if d not in (None, "") else 0.0

qr = df[df["room_type"].isin(ROOM_TYPES) & df["price_quote_raw"].notna()].copy()
qr["has_discount"] = qr["price_quote_raw"].map(discount_amount) > 0
print(qr.groupby(["room_type", qr["quote_nights"].clip(upper=10)])["has_discount"]
        .mean().round(3).unstack(0))

room_type     Entire home/apt  Private room
quote_nights                               
1.0                     0.263         0.164
2.0                     0.192         0.109
3.0                     0.176         0.108
4.0                     0.149         0.076
5.0                     0.140         0.091
6.0                     0.126         0.081
7.0                     0.550         0.548
8.0                     0.615         0.714
9.0                     0.529         0.538
10.0                    0.565         0.552


## 3. 필터링

각 단계에서 몇 개가 빠지는지 표로 남긴다.

### 판단: 이름에 운영 중단이 표시된 숙소 제외 (6개)

- 가격과 리뷰가 있어도 운영 중이 아닌 숙소가 있다. 숙소 이름에서 운영 중단을 뜻하는 표현을 검색한 뒤 직접 읽어 확인한 6개를 id로 제외했다 ("No Longer Available", "Not available until 2022", "let out full time" 등).
- 이름에 표시하지 않은 비활성 숙소는 여전히 남아 있을 수 있으며, 이는 한계로 기록한다.


### 판단: 가격 상·하위 1% 제외

- 극단적인 가격은 적정 가격 모델을 왜곡하므로, 다른 필터를 모두 적용한 **마지막**에 상·하위 1%를 제외한다. 순서를 바꾸면 자르는 기준값이 달라진다.
- **상위 1%:** 2인용 원룸 £10,000 등 실제 판매 가격이 아니라 예약을 막기 위해 설정한 것으로 보이는 가격이 포함되어 있다. 이는 "바가지"가 아니라 사실상 판매 중지 상태이므로 제외한다.
- **하위 1%:** 대부분 외곽 지역의 1인용 개인실로, 실제로 싼 숙소로 보인다. 오류는 아니지만 적정 가격 모델이 같은 조건의 싼 가격을 학습하므로, 제외해도 결론에 미치는 영향은 작다고 판단한다.


### 판단: 예약 가능일 1일 이상

- `availability_365`가 0인 숙소는 닫힌 숙소일 수도 있지만 1년 내내 예약이 찬 숙소일 수도 있다. 제외되는 숫자가 작아(위 표) 결과 영향은 작다고 판단한다.

### 판단: 숙소 유형은 집 전체·개인실만 사용 (409개 제외)

| 숙소 유형 | 숙소 수 | 비율 |
|---|---|---|
| 집 전체 | 61,074 | 66.4% |
| 개인실 | 30,471 | 33.1% |
| 호텔룸 | 207 | 0.2% |
| 다인실 | 202 | 0.2% |

- **표본이 너무 적다:** 호텔룸과 다인실은 합쳐서 0.4%뿐이다. 리뷰·가격 필터까지 거치면 더 줄어, 적정 가격을 안정적으로 계산하기 어렵다.
- **가격 구조가 다르다:** 호텔룸은 숙박업체가 운영하는 객실이고, 다인실은 방 하나를 여러 명이 나눠 쓰는 침대 단위 가격이다. 집 전체·개인실과 같은 기준으로 "적정 가격"을 비교하기 어렵다.
- **의사결정 대상과 맞다:** 호스트 성장팀의 가격 안내 대상은 대부분 집 전체·개인실을 운영하는 일반 호스트다.
- 참고: `property_type`이 "Room in hotel"이지만 호스트가 개인실로 등록한 숙소는 분석에 남아 있다.

In [8]:
# 숙소 유형별 숙소 수와 비율 (필터 전, 전체 데이터)
room_counts = pd.DataFrame({
    "숙소 수": df["room_type"].value_counts(),
    "비율(%)": (df["room_type"].value_counts(normalize=True) * 100).round(1),
})
print("전체 숙소 수:", len(df))
room_counts

전체 숙소 수: 91954


,숙소 수,비율(%)
room_type,,
Entire home/apt,61074,66.4
Private room,30471,33.1
Hotel room,207,0.2
Shared room,202,0.2


In [9]:
# 리뷰 수 구간별 가성비 점수가 얼마나 흔들리는지 (가격 있는 집 전체·개인실)
rv = df[df["room_type"].isin(ROOM_TYPES) & df["price_gbp"].notna() & df["review_scores_value"].notna()]
bins = pd.cut(rv["number_of_reviews"], [0, 2, 4, 9, 19, 49, 100000],
              labels=["1~2", "3~4", "5~9", "10~19", "20~49", "50+"])
g = rv.groupby(bins, observed=True)["review_scores_value"]

stability = pd.DataFrame({
    "숙소 수": g.size(),
    "점수 흔들림(표준편차)": g.std().round(2),
    "5.0 만점 비율(%)": (g.apply(lambda x: (x == 5).mean()) * 100).round(1),
    "4점 미만 비율(%)": (g.apply(lambda x: (x < 4).mean()) * 100).round(1),
})
stability

,숙소 수,점수 흔들림(표준편차),5.0 만점 비율(%),4점 미만 비율(%)
number_of_reviews,,,,
1~2,7822,1.02,60.2,16.2
3~4,4518,0.58,38.1,12.4
5~9,6895,0.40,20.7,7.2
10~19,7616,0.28,7.4,2.5
20~49,10156,0.22,1.7,0.8
50+,10429,0.18,0.2,0.2


In [10]:
steps = []
def log(name, d):
    steps.append({"단계": name, "숙소 수": len(d)})
    return d

s = log("전체", df)
s = log("집 전체·개인실", s[s["room_type"].isin(ROOM_TYPES)])
s = log(f"리뷰 {MIN_REVIEWS}개 이상", s[s["number_of_reviews"] >= MIN_REVIEWS])
s = log("가격 있음", s[s["price_gbp"].notna()])
s = log(f"예약 가능일 {MIN_AVAILABILITY}일 이상", s[s["availability_365"] >= MIN_AVAILABILITY])
s = log(f"조회 기간 {MAX_QUOTE_NIGHTS}박 이하", s[s["quote_nights"] <= MAX_QUOTE_NIGHTS])
s = log("운영 중단 표시 숙소 제외", s[~s["id"].isin(INACTIVE_IDS)])

lo, hi = s["price_gbp"].quantile([TRIM_Q, 1 - TRIM_Q])
s = log(f"가격 상·하위 {TRIM_Q:.0%} 제외 (£{lo:.0f}~£{hi:.0f})", s[s["price_gbp"].between(lo, hi)])

filter_table = pd.DataFrame(steps)
filter_table["제외"] = -filter_table["숙소 수"].diff().fillna(0).astype(int)
filter_table

,단계,숙소 수,제외
0,전체,91954,0
1,집 전체·개인실,91545,409
2,리뷰 10개 이상,38504,53041
3,가격 있음,28201,10303
4,예약 가능일 1일 이상,27923,278
5,조회 기간 6박 이하,26124,1799
6,운영 중단 표시 숙소 제외,26118,6
7,가격 상·하위 1% 제외 (£40~£958),25607,511


## 4. 피처 만들기

**목적:** 적정 가격 모델에 넣을 재료를 원본에서 숫자로 만든다.

**한 일**

| 새 열 | 만든 방법 | 이유 |
|---|---|---|
| `bath_n` | `bathrooms_text`("1.5 baths" 등)에서 숫자 추출, "Half-bath"는 0.5 | `bathrooms` 열이 전부 비어 있음 |
| `bath_shared` | 욕실 설명에 "shared"가 있으면 1 | 공용 욕실 숙소가 "숨은 보석"으로 잘못 분류되는 것 방지 |
| `amenities_n` | 편의시설 목록(JSON 문자열)의 개수 | 목록 그대로는 모델에 넣을 수 없음 |
| `dist_center_km` | Charing Cross까지 거리 (하버사인 공식) | 자치구(33개)보다 세밀한 위치 차이 반영 |
| `log_price` | 가격의 로그 | 비싼 쪽으로 치우친 분포를 펴고, 비율(%) 차이로 해석 |
| `quote_month` | 가격 조회 체크인 월 | 성수기 요금 차이 반영 |

**결과** (25,607개 숙소)

| | 욕실 수 | 공용 욕실 | 편의시설 수 | 도심 거리 |
|---|---|---|---|---|
| 중앙값 | 1개 | 19% (평균) | 37개 | 5.7km |
| 범위 | 0~9개 | | 0~99개 | 0~28.8km |

- 욕실 수 18개는 숫자를 추출하지 못해 5절에서 채운다.

In [11]:
s = s.copy()

# 욕실 수: "1.5 baths", "1 shared bath", "Half-bath" 등
bt = s["bathrooms_text"].fillna("").str.lower()
s["bath_n"] = bt.str.extract(r"([\d.]+)")[0].astype(float)
s.loc[s["bath_n"].isna() & bt.str.contains("half"), "bath_n"] = 0.5
s["bath_shared"] = bt.str.contains("shared").astype(int)

# 편의시설 개수: JSON 목록 문자열
s["amenities_n"] = s["amenities"].map(lambda x: len(json.loads(x)) if isinstance(x, str) else np.nan)

# 도심까지 거리(km): 하버사인 공식
def haversine_km(lat, lon, lat0, lon0):
    lat, lon, lat0, lon0 = map(np.radians, [lat, lon, lat0, lon0])
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))
s["dist_center_km"] = haversine_km(s["latitude"], s["longitude"], *CENTER)

s["log_price"] = np.log(s["price_gbp"])
s["quote_month"] = pd.to_datetime(s["price_quote_checkin_date"]).dt.month

print(s[["bath_n", "bath_shared", "amenities_n", "dist_center_km"]].describe().round(2))

         bath_n  bath_shared  amenities_n  dist_center_km
count  25589.00     25607.00     25607.00        25607.00
mean       1.30         0.19        36.28            7.04
std        0.59         0.39        13.14            4.79
min        0.00         0.00         0.00            0.01
25%        1.00         0.00        27.00            3.63
50%        1.00         0.00        37.00            5.71
75%        1.50         0.00        45.00            9.53
max        9.00         1.00        99.00           28.77


In [12]:
# 조회 월 분포 (최종 분석 대상 전체)
print("조회 월별 숙소 비율:")
print(s["quote_month"].value_counts(normalize=True).sort_index().round(3))
print("9~10월 비율:", round(s["quote_month"].isin([9, 10]).mean(), 3))

# 조회 월별 1박 가격 중앙값 (같은 조건끼리: 침실 1개짜리 집 전체)
e1 = s[(s["room_type"] == "Entire home/apt") & (s["bedrooms"] == 1)]
print(e1.groupby("quote_month")["price_gbp"].agg(["median", "size"]).round(0))

조회 월별 숙소 비율:
quote_month
1     0.042
2     0.009
3     0.007
4     0.004
5     0.004
6     0.004
7     0.003
8     0.002
9     0.591
10    0.252
11    0.049
12    0.033
Name: proportion, dtype: float64
9~10월 비율: 0.843
             median  size
quote_month              
1             179.0   383
2             168.0    55
3             155.0    54
4             155.0    37
5             163.0    32
6             157.0    25
7             199.0    23
8             175.0    21
9             171.0  4217
10            182.0  2156
11            172.0   402
12            183.0   267


## 5. 결측 처리

- 모델 재료 13개 중 결측: 침대 수 481개(1.9%), 욕실 수 18개(0.1%), 침실 수 13개(0.05%)
- 같은 숙소 유형·수용 인원 숙소들의 중앙값으로 채웠다. 비율이 작고, 비슷한 숙소 기준이라 왜곡이 작다고 판단한다.
- 가격과 달리 이 값들은 모델의 재료일 뿐이라, 지우기보다 채워서 숙소를 남기는 편이 낫다.

In [13]:
feature_cols = ["neighbourhood_cleansed", "room_type", "accommodates", "bedrooms", "beds",
                "bath_n", "bath_shared", "amenities_n", "dist_center_km",
                "latitude", "longitude", "quote_nights", "quote_month"]

missing = pd.DataFrame({
    "결측 수": s[feature_cols].isna().sum(),
    "결측 비율(%)": (s[feature_cols].isna().mean() * 100).round(2),
})
print("전체 숙소 수:", len(s))
missing[missing["결측 수"] > 0].sort_values("결측 수", ascending=False)

전체 숙소 수: 25607


,결측 수,결측 비율(%)
beds,481,1.88
bath_n,18,0.07
bedrooms,13,0.05


In [14]:
# 침대·욕실 결측은 같은 숙소 유형·수용 인원의 중앙값으로 채운다
for col in ["beds", "bath_n", "bedrooms"]:
    n_missing = s[col].isna().sum()
    s[col] = s[col].fillna(s.groupby(["room_type", "accommodates"])[col].transform("median"))
    s[col] = s[col].fillna(s[col].median())
    print(f"{col}: {n_missing}개 채움")

beds: 481개 채움
bath_n: 18개 채움
bedrooms: 13개 채움


In [15]:
col = "estimated_revenue_l365d"

# 1) 원본 데이터에 있는지
print("원본 df에 있음:", col in df.columns)

# 2) 정리 파일에 있는지
clean = pd.read_csv("listings_clean.csv")
print("정리 파일에 있음:", col in clean.columns)

# 3) 있으면 값 확인
if col in df.columns:
    print(df[col].describe().round(0))

원본 df에 있음: True


FileNotFoundError: [Errno 2] No such file or directory: 'listings_clean.csv'

## EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(s["price_gbp"], bins=60)
axes[0].set_title("1박 가격 분포 (£)")
axes[1].hist(s["log_price"], bins=60)
axes[1].set_title("로그 가격 분포")
plt.tight_layout(); plt.show()

: 

In [ ]:
borough = (s.groupby("neighbourhood_cleansed")["price_gbp"].median().sort_values())
fig, ax = plt.subplots(figsize=(8, 9))
borough.plot.barh(ax=ax)
ax.set_title("자치구별 1박 가격 중앙값 (£)")
ax.set_ylabel("")
plt.tight_layout(); plt.show()

NameError: name 's' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(s["review_scores_value"], bins=40)
ax.set_title("가성비 점수 분포")
plt.tight_layout(); plt.show()
print("4.5점 이상 비율:", round((s["review_scores_value"] >= 4.5).mean(), 3))
print(s["review_scores_value"].describe().round(2))